# 1.1 Reading a 2D scalar field from a simulation

First we define the objects we will use - the mesh reader (to read the mesh from the FEM file), the pickle manager (to save the mesh data that we want to analyse), and the unfolder (to manage the mesh).

In [1]:
# 1. Imports and Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
from cyclops import sensors, sensor_suite, experiment, pyomo_problem  # Replace with actual module imports
from cyclops.sim_reader import MeshReader
from cyclops.object_reader import PickleManager
from cyclops.optimisers import NSGA2Optimiser
from cyclops.regressors import LModel, CSModel, RBFModel
from cyclops.fields import VectorField, ScalarField, Field

In [2]:
# 2. Load Mesh and Setup Test Data
# Assuming mesh is a 2D or 3D array of points (could be read from a file or generated here)

reader = MeshReader("/home/cbyers/Projects/cyclops/tutorials/data/monoblock_out.e")
pickle_manager = PickleManager()
point_dict = reader.point_data

print(point_dict.keys())
print(reader.read_region_names())

# Read the simulation data
sensor_region = "right"
pos_3D = reader.read_pos(sensor_region)
grid = reader.generate_grid(5)

# Get the min and max bounds along each axis (x, y, z)
min_bounds = np.min(grid, axis=0)
max_bounds = np.max(grid, axis=0)
bounds = np.vstack((min_bounds, max_bounds))

disp = np.array(
    [
        reader.read_scalar(sensor_region, "disp_x"),
        reader.read_scalar(sensor_region, "disp_y"),
        reader.read_scalar(sensor_region, "disp_z"),
    ]
).T

disp_field = VectorField(RBFModel, bounds)
print(disp_field.get_shape())
disp_field.fit_model(pos_3D, disp)
print(disp_field.get_shape())
print(type(reader))


dict_keys(['disp_x', 'disp_y', 'disp_z', 'temperature'])
['right', 'top', 'left', '', 'centre_x_bottom_y_back_z', 'centre_x_bottom_y_front_z', 'left_x_bottom_y_centre_z', 'right_x_bottom_y_centre_z']
(2, 3)
vector_values  [[ 5.42377424e-05  3.28911344e-05 -2.36492004e-05]
 [ 5.43988560e-05  3.49743114e-05 -1.95260568e-05]
 [ 5.43301892e-05  3.39904317e-05 -2.15822030e-05]
 ...
 [ 9.36166119e-05  7.56636920e-05  4.08665791e-05]
 [ 9.10786507e-05  7.06711488e-05  4.32006640e-05]
 [ 9.11093409e-05  7.20019583e-05  3.95953852e-05]]
(2, 3)
<class 'cyclops.sim_reader.MeshReader'>


In [ ]:
reader = MeshReader("/home/cbyers/Projects/cyclops/tutorials/data/monoblock_out.e")
new_experiment = experiment.Experiment(
    reader=reader,
    no_sensors=2,
    sensor_types=["PointSensor", "RoundSensor"],
    field_regressor=['RBFModel','RBFModel'],
    field_positions=pos_3D,
    optimiser=NSGA2Optimiser,
    field_type="VectorField",
    noise_list=(0.2, 0.1)
)

set_name  ['right', 'top', 'left', '', 'centre_x_bottom_y_back_z', 'centre_x_bottom_y_front_z', 'left_x_bottom_y_centre_z', 'right_x_bottom_y_centre_z']
s  right
s  top
s  left
s  
s  centre_x_bottom_y_back_z
s  centre_x_bottom_y_front_z
s  left_x_bottom_y_centre_z
s  right_x_bottom_y_centre_z


ValueError: Invalid field_name format. Must be a string (scalar) or list (vector).

In [ ]:
new_field = VectorField.mesh_reader_init(mesh_reader=reader, set_name="right",
                                         field_name=["disp_x", "disp_y", "disp_z"], regression_type=RBFModel)

In [ ]:
# 3. Test Sensor Class
# 3.1 Test PointSensor
point_sensor = sensors.PointSensor(
    offset_function=lambda x: 0.1 * np.random.randn(*x.shape),
    centre_point=np.array([0.5, 0.5, 0.5]),
    field_dim=3,
    field=disp_field,
    noise_dev=0.01,
    failure_chance=0.05
)

psensor_sites = point_sensor.get_measurement_sites()
print(point_sensor.get_measurement_sites())
print(point_sensor.get_failure_chance())

In [ ]:
# 3.2 Test RoundSensor
round_sensor = sensors.RoundSensor(
    offset_function=lambda x: 0.1 * np.random.randn(*x.shape),
    field_dim=3,
    field=disp_field,
    norm_vector=np.array([1, 1, 1]),
    centre_point=np.array([0.5, 0.5, 0.5]),
    radius=0.2,
    noise_dev=0.01,
    failure_chance=0.05
)

rsensor_sites = round_sensor.get_measurement_sites()
print(round_sensor.get_measurement_sites())
print(round_sensor.get_input_sites())


In [ ]:

# 3.3 Get sensor output for the mesh points
point_sensor_readings, point_sensor_positions = point_sensor.get_output_values(
    true_site_values=disp_field, actual_pos=psensor_sites
)

round_sensor_readings, round_sensor_positions = round_sensor.get_output_values(
    true_site_values=disp_field, actual_pos=rsensor_sites
)


In [ ]:
# 4. **Test SensorSuite Class**
layout_one = sensor_suite.SensorSuite(
    true_field=disp_field, 
    sensors=[point_sensor, round_sensor],
    #sensor_pos=[np.array([0.5, 0.5, 0.5]), np.array([0.6, 0.6, 0.6])],
    field_points=pos_3D,  # Mesh points
    field_vec=disp_field  # Field values at mesh points
)

# Get sensor sites (test data collection)
sensor_observations, sensor_sites = layout_one.get_sensor_outputs()
print(sensor_observations)
print(sensor_sites)


In [ ]:

# Visualize the sensor positions and the field (if relevant)
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Use a colormap to generate distinct colors
cmap = plt.cm.get_cmap("viridis", len(sensor_sites))

# Extract x, y, z coordinates
x = pos_3D[:, 0]
y = pos_3D[:, 1]
z = pos_3D[:, 2]

# Create 3D figure
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Wireframe plot of the mesh (connecting points by lines)
for i in range(0, pos_3D.shape[0] - 1):
    ax.plot([pos_3D[i, 0], pos_3D[i + 1, 0]],
            [pos_3D[i, 1], pos_3D[i + 1, 1]],
            [pos_3D[i, 2], pos_3D[i + 1, 2]],
            color='blue', alpha=0.9)

# Loop through each sensor group
for i, group in enumerate(sensor_sites):
    color = cmap(i)  # Assign a unique color from the colormap

    # Check if it's a single point or multiple points
    if group.ndim == 1:  # Single point
        ax.scatter(*group, color=color, marker='o', s=100, label=f'Sensor {i+1} (Single Point)')
    else:  # Multiple points
        ax.scatter(group[:, 0], group[:, 1], group[:, 2], 
                   color=color, marker='^', s=60, label=f'Sensor {i+1} (Multiple Points)')


# Loop through each sensor group
for i, group in enumerate(sensor_sites):
    color = cmap(i)  # Assign a unique color from the colormap

    # Check if it's a single point or multiple points
    if group.ndim == 1:  # Single point
        ax.scatter(*group, color=color, marker='o', s=100, label=f'Sensor {i+1} (Single Point)')
    else:  # Multiple points
        ax.scatter(group[:, 0], group[:, 1], group[:, 2], 
                   color=color, marker='^', s=60, label=f'Sensor {i+1} (Multiple Points)')


# Labels and title
ax.set_xlabel('X Axis')
ax.set_ylabel('Y Axis')
ax.set_zlabel('Z Axis')
ax.set_title('3D Scatter Plot')

# Show plot
plt.show()


In [ ]:
# Create 3D figure
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

plt.scatter(pos_3D[:, 0], pos_3D[:, 1], pos_3D[:, 2], c=disp_field)
plt.colorbar(label="Field Value")
plt.title("Field with Sensor Positions")
plt.scatter(point_sensor_positions[:, 0], point_sensor_positions[:, 1], color="red", label="PointSensor")
plt.scatter(round_sensor_positions[:, 0], round_sensor_positions[:, 1], color="green", label="RoundSensor")
plt.legend()
plt.show()

# Visualize sensor data
plt.scatter(sensor_sites[:, 0], sensor_sites[:, 1], color="blue", label="Sensor Sites")
plt.colorbar(label="Field Value")
plt.title("Sensor Sites and Observations")
plt.legend()
plt.show()


In [ ]:
from cyclops.optimisers import MOOProblem, Optimiser
from cyclops.pyomo_problem import SensorPlacementOptimisation
# 5. **Test Optimiser Class**
# Define a Pyomo model
snsr_optimiser = SensorPlacementOptimisation(reader.__mesh, num_sensors=2, sensor_types=[''])



In [ ]:
class DummyModel:
    def update_sensor_positions(self, positions):
        self.positions = positions

    def solve(self):
        return np.random.rand(), np.random.rand()  # Dummy MSE and risk

# Create an MOOProblem instance
pyomo_model = DummyModel()
moo_problem = MOOProblem(pyomo_model, num_sensors=2)  # Example: 2 sensors

# Instantiate the Optimiser
optimiser = Optimiser(time_limit="10m", algorithm="NSGA2")  # Example: NSGA2 as the algorithm

# Run the optimizer
result = optimiser.optimise(moo_problem)

# Display results
print("Optimization Result:")
print(result)

# Optionally: visualize optimization progress (if available from result)
plt.plot(result.F[:, 0], result.F[:, 1], 'o')
plt.title("Optimization Progress (MSE vs Risk)")
plt.xlabel("MSE")
plt.ylabel("Risk")
plt.show()

